# Exercise 4

In [27]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import math

## 1

In [ ]:
lamb = 1
m_servers = 10
mean_service_time = 8
mean_time_between_customers = 1
customers = 10000
n_reps = 10

In [29]:
def confidence_interval(data, alpha = 0.05):
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_score = stats.t.ppf(1 - alpha/2, df=n-1)
    margin_of_error = t_score * std_err
    return mean, mean - margin_of_error, mean + margin_of_error

In [33]:
def theoretical_erlang_block(A,m):
    num = (A**m) / math.factorial(m)
    denom = sum((A**k) / math.factorial(k) for k in range(m+1))
    return num / denom

In [34]:
def simulate_queue(arrivals_distribution, service_distribution, customers, m_serversm, n_reps):
    blocking_fractions = []
    for _ in range(n_reps):
        inbetween_arrivals = arrivals_distribution(customers)
        service_times = service_distribution(customers)
        arrival_times = np.cumsum(inbetween_arrivals)

        service_free_times = np.zeros(m_servers)
        blocked = 0

        for i in range(customers):
            arr = arrival_times[i]

            free_servers = service_free_times <= arr

            if np.any(free_servers):
                server_index = np.argmax(free_servers)
                service_free_times[server_index] = arr + service_times[i]
            else:
                blocked += 1
        
        blocking_fraction = blocked / customers
    
        blocking_fractions.append(blocking_fraction)
    
    mean, lower, upper = confidence_interval(blocking_fractions)
    return mean, lower, upper


In [36]:
np.random.seed(42)  # For reproducibility
def arr_poisson(customers):
    return np.random.exponential(mean_time_between_customers, size=customers)

def service_exp(customers):
    return np.random.exponential(mean_service_time, size=customers)


mean_pos, lower_pos, upper_pos = simulate_queue(arr_poisson, service_exp, customers, m_servers,n_reps)

print(f"Simulated Blocking Fraction: {mean_pos:.4f}")
print(f"95% Confidence Interval: ({lower_pos:.4f}, {upper_pos:.4f})")

print(f"Theoretical Blocking Fraction (Erlang B): {theoretical_erlang_block(lamb * mean_service_time, m_servers):.4f}")

Simulated Blocking Fraction: 0.1217
95% Confidence Interval: (0.1200, 0.1234)
Theoretical Blocking Fraction (Erlang B): 0.1217


## part 2

In [37]:
np.random.seed(42)  # For reproducibility
p1 = 0.8
lambda1 = 0.8333
p2 = 0.2
lambda2 = 5

def arr_erlang(customers):
    return np.random.gamma(shape=2, scale=mean_time_between_customers/2, size=customers)

def arr_hyperexp(customers):
    coin = np.random.rand(customers)

    return np.where( coin >= p1, 
                    np.random.exponential(1/lambda1, size=customers), 
                    np.random.exponential(1/lambda2, size=customers))

mean_erlang, lower_erlang, upper_erlang = simulate_queue(arr_erlang, service_exp, customers, m_servers,n_reps)
mean_hyperexp, lower_hyperexp, upper_hyperexp = simulate_queue(arr_hyperexp, service_exp, customers, m_servers,n_reps)

print(f"Simulated Blocking Fraction (Erlang Arrivals): {mean_erlang:.4f}")
print(f"95% Confidence Interval (Erlang Arrivals): ({lower_erlang:.4f}, {upper_erlang:.4f})")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Hyperexponential Arrivals): {mean_hyperexp:.4f}")
print(f"95% Confidence Interval (Hyperexponential Arrivals): ({lower_hyperexp:.4f}, {upper_hyperexp:.4f})") 

Simulated Blocking Fraction (Erlang Arrivals): 0.0933
95% Confidence Interval (Erlang Arrivals): (0.0918, 0.0947)
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Hyperexponential Arrivals): 0.5632
95% Confidence Interval (Hyperexponential Arrivals): (0.5607, 0.5656)


## part 3

In [38]:
np.random.seed(42)  # For reproducibility

def service_constant(customers):
    return np.full(customers, mean_service_time)

def service_pareto(customers, k):
    beta = mean_service_time * (k - 1) / k

    return (np.random.pareto(k, size=customers) + 1) * beta

def service_erlang(customers):
    return np.random.gamma(shape=4, scale=mean_service_time/2, size=customers)

def service_uniform(customers):
    return np.random.uniform(low=0.0, high=2*mean_service_time, size=customers)

mean_const, lower_const, upper_const = simulate_queue(arr_poisson, service_constant, customers, m_servers,n_reps)

mean_p1, lower_p1, upper_p1 = simulate_queue(arr_poisson, lambda c: service_pareto(c, 1.05), customers, m_servers,n_reps)
mean_p2, lower_p2, upper_p2 = simulate_queue(arr_poisson, lambda c: service_pareto(c, 2.05), customers, m_servers,n_reps)

mean_erlang, lower_erlang, upper_erlang = simulate_queue(arr_poisson, service_erlang, customers, m_servers,n_reps)
mean_uniform, lower_uniform, upper_uniform = simulate_queue(arr_poisson, service_uniform, customers, m_servers,n_reps)

print(f"Theoretical Blocking Fraction (Erlang B): {theoretical_erlang_block(lamb * mean_service_time, m_servers):.4f}")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Constant Service): {mean_const:.4f}")
print(f"95% Confidence Interval (Constant Service): ({lower_const:.4f}, {upper_const:.4f})")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Pareto k=1.05 Service): {mean_p1:.4f}")
print(f"95% Confidence Interval (Pareto k=1.05 Service): ({lower_p1:.4f}, {upper_p1:.4f})")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Pareto k=2.05 Service): {mean_p2:.4f}")
print(f"95% Confidence Interval (Pareto k=2.05 Service): ({lower_p2:.4f}, {upper_p2:.4f})")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Erlang Service): {mean_erlang:.4f}")
print(f"95% Confidence Interval (Erlang Service): ({lower_erlang:.4f}, {upper_erlang:.4f})")
print("--------------------------------------------------------------------------------------")
print(f"Simulated Blocking Fraction (Uniform Service): {mean_uniform:.4f}")
print(f"95% Confidence Interval (Uniform Service): ({lower_uniform:.4f}, {upper_uniform:.4f})")

Theoretical Blocking Fraction (Erlang B): 0.1217
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Constant Service): 0.1213
95% Confidence Interval (Constant Service): (0.1198, 0.1229)
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Pareto k=1.05 Service): 0.0014
95% Confidence Interval (Pareto k=1.05 Service): (0.0009, 0.0018)
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Pareto k=2.05 Service): 0.1198
95% Confidence Interval (Pareto k=2.05 Service): (0.1171, 0.1224)
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Erlang Service): 0.4407
95% Confidence Interval (Erlang Service): (0.4389, 0.4425)
--------------------------------------------------------------------------------------
Simulated Blocking Fraction (Uniform